# Exploratory Data Analysis & Trust Assessment
### Asymmetric Cross-Source Trust Assessment of Single-Meter Load Measurements for Chance-Constrained Battery Dispatch

This notebook provides interactive exploration of:
1. Synchronized multi-channel dataset (Load, Ambient Temperature, Day-Ahead Price, Renewable Generation).
2. Chronological data partitions (Train 60%, Calibration 20%, Test 20%).
3. Dual-quantile regression model predictions and non-crossing intervals.
4. Injected attack scenarios (S1, S2, S3, S4).
5. Asymmetric suspicion scoring dynamics and battery dispatch outcomes.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project src is in Python path
sys.path.insert(0, os.path.abspath('../src'))

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

## 1. Load Synchronized Dataset (26,304 Hourly Observations)

In [ ]:
train_df = pd.read_csv('../data/processed/train.csv', index_col=0, parse_dates=True)
cal_df = pd.read_csv('../data/processed/cal.csv', index_col=0, parse_dates=True)
test_df = pd.read_csv('../data/processed/test.csv', index_col=0, parse_dates=True)
full_df = pd.concat([train_df, cal_df, test_df])

print(f"Total synchronized observations: {len(full_df)}")
print(f"Train: {len(train_df)}, Calibration: {len(cal_df)}, Test: {len(test_df)}")
full_df.head()

## 2. Multi-Channel Time-Series Visualization

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

axes[0].plot(full_df.index, full_df['load'], color='#1d3557', lw=0.8)
axes[0].set_ylabel('Load (MW)')
axes[0].set_title('Three-Year Synchronized Operational Time-Series (2018 - 2020)')

axes[1].plot(full_df.index, full_df['temperature'], color='#e76f51', lw=0.8)
axes[1].set_ylabel('Temperature (°C)')

axes[2].plot(full_df.index, full_df['price'], color='#2a9d8f', lw=0.8)
axes[2].set_ylabel('Day-Ahead Price ($/MWh)')

axes[3].plot(full_df.index, full_df['renewable'], color='#e9c46a', lw=0.8)
axes[3].set_ylabel('Renewable Gen (MW)')
axes[3].set_xlabel('Date')

# Highlight split boundaries
for ax in axes:
    ax.axvline(cal_df.index[0], color='gray', ls='--', label='Train/Cal Split' if ax==axes[0] else "")
    ax.axvline(test_df.index[0], color='black', ls='--', label='Cal/Test Split' if ax==axes[0] else "")

axes[0].legend(loc='upper right')
plt.tight_layout()
plt.show()

## 3. Cross-Channel Correlation Analysis

In [ ]:
corr = full_df.corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45)
ax.set_yticklabels(corr.columns)
plt.colorbar(im, ax=ax)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", color="black")
plt.title('Empirical Channel Cross-Correlation Heatmap')
plt.tight_layout()
plt.show()

## 4. Inspection of Published Results

In [ ]:
t6 = pd.read_csv('../results/tables/table_6_forecasting_accuracy.csv')
print("=== TABLE 6: FORECASTING ACCURACY ===")
display(t6)

t8 = pd.read_csv('../results/tables/table_8_detection_performance.csv')
print("\n=== TABLE 8: DETECTION PERFORMANCE ===")
display(t8)

t10 = pd.read_csv('../results/tables/table_10_suspicion_ablation.csv')
print("\n=== TABLE 10: SUSPICION SCORE ABLATION ===")
display(t10)